# EDEN-07 — Climate and vision alignment | v1.0

**Original plan: Phase 2, Notebook 07.** Connect the verified climate/control tables to the latest usable image information, without looking into the future. Build forecast-window indices and fit normalization on training data only. This notebook prepares data; it does not train or evaluate a forecast model.

**Kaggle:** CPU, Internet **Off**. Attach only **EDEN-01 v1.1 results** (`results_1.1.zip` or its extracted folder) and **EDEN-06 v1.0 results**. No raw image dataset, GPU, or new download is needed. Run all cells in order. Return `EDEN-07-Climate-Vision-Alignment-Results-v1.0.zip`.

**What 06 established:** 28 result hashes verified; all 24 validation metrics recomputed from saved predictions; 311 out-of-fold rows; 32/36 usable canopy images. RGB-D improved the leaf-area validation MAE over RGB-only, but RGB-only was better on the other three traits. Some red-fruit predictions were negative. The 2024 canopy traits are still unvalidated transfer estimates.

**How 07 handles that:** keep image embeddings (compact learned descriptions) as optional experimental inputs; keep missingness, age, and quality visible. Physical trait proxies and the unreliable green-mask measurements are audit columns, excluded from default model inputs. The 2023 OOF table is not joined to 2024 climate timestamps.

**Time rule:** use 72 five-minute readings ending at the forecast origin. Their first and last timestamps are 355 minutes apart. Images must be within that same history and the same evaluation split. This conservative image tolerance follows the existing history contract; it is not tuned to scores. Sparse photos will leave most rows without vision, which is expected and must remain explicit.

**Outputs:** `data/aligned.parquet`, `data/window_index.parquet`, `data/windows.json`, `data/scaler.json`, split/feature contracts, coverage reports, and a hash manifest. Original Notebook 08 will use these for matched climate baselines at **15, 30, 60, 180, and 360 minutes**.

Reference: original EDEN SRS & Implementation Handbook, module 07 and temporal-model contract. [Backward joins in pandas](https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html). Sources: [2024 challenge](https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1) and [2023 pre-trial](https://doi.org/10.4121/e1ee9de9-6ce9-4502-a37c-34b5b1372bed.v1), CC BY 4.0.

## Cell 1 — Settings and folders

An origin is the moment a forecast starts. A horizon is how far ahead it predicts. We keep the original six-hour history limit and prepare all five planned horizons.

In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib, io, json, zipfile, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ROOT = WORK / 'eden07'
RESULT = WORK / 'EDEN-07-Climate-Vision-Alignment-Results-v1.0.zip'
LOOKBACK_STEPS = 72
HORIZON_MINUTES = [15, 30, 60, 180, 360]
TARGETS = ['air_temp_c', 'rh_pct', 'co2_ppm']
KEYS = ['experiment_id', 'compartment_id', 'ts_utc']
for folder in ['data', 'reports']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

def sha(payload):
    """Fingerprint the exact bytes of a file."""
    return hashlib.sha256(payload).hexdigest()

def save_json(path, value):
    """Write readable metadata, rejecting NaN or infinity in JSON reports."""
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')

versions = {name: importlib.metadata.version(name) for name in ['numpy', 'pandas', 'pyarrow', 'matplotlib']}
save_json(ROOT / 'reports/runtime.json', versions)
print('CPU alignment; no model training or network downloads.')

## Cell 2 — Find and verify EDEN-01 and EDEN-06

Check the receipt for each input, then check every file against its recorded size and fingerprint. ZIP and extracted Kaggle inputs both work. Duplicate copies with identical manifests are harmless; different runs are rejected.

In [ ]:
class Source:
    def __init__(self, path, member=None):
        self.path, self.member = Path(path), member
        if member is None:
            self.manifest_bytes = self.path.read_bytes()
            self.base = self.path.parent
        else:
            with zipfile.ZipFile(self.path) as archive:
                self.manifest_bytes = archive.read(member)
            self.prefix = str(PurePosixPath(member).parent)
        self.manifest = json.loads(self.manifest_bytes)

    def read(self, relative):
        # Accept only paths relative to the producer's artifact directory.
        relative = str(relative).replace(chr(92), '/')
        parts = PurePosixPath(relative)
        assert not parts.is_absolute() and '..' not in parts.parts and ':' not in relative
        if self.member is None:
            target = (self.base / relative).resolve()
            assert target.is_relative_to(self.base.resolve())
            return target.read_bytes()
        name = str(PurePosixPath(self.prefix) / parts)
        with zipfile.ZipFile(self.path) as archive:
            return archive.read(name)



needed = {'EDEN-01': ('1.1', 'PASS_ETL'),
          'EDEN-06': ('1.0', 'PASS_VISION_PIPELINE_TRANSFER_UNVALIDATED')}
candidates = {producer: {} for producer in needed}

def consider(source):
    """Recognize only the producers and versions this notebook understands."""
    producer = source.manifest.get('producer')
    if producer in needed and str(source.manifest.get('code_version')) == needed[producer][0]:
        candidates[producer][sha(source.manifest_bytes)] = source

for path in sorted(INPUT.rglob('*.zip')):
    try:
        with zipfile.ZipFile(path) as bundle:
            names = [n for n in bundle.namelist() if PurePosixPath(n).name == 'manifest.json']
        for name in names:
            consider(Source(path, name))
    except (zipfile.BadZipFile, json.JSONDecodeError, UnicodeDecodeError):
        continue
for path in sorted(INPUT.rglob('manifest.json')):
    try:
        consider(Source(path))
    except (json.JSONDecodeError, UnicodeDecodeError):
        continue
sources, verification = {}, []
for producer, (_, status) in needed.items():
    assert len(candidates[producer]) == 1, f'Attach exactly one distinct {producer} result.'
    source = next(iter(candidates[producer].values()))
    assert source.manifest['status'] == status, f'Unexpected status for {producer}'
    files = source.manifest['files']
    assert files and len(files) == len({item['path'] for item in files})
    for item in files:
        payload = source.read(item['path'])
        assert len(payload) == item['size'] and sha(payload) == item['sha256'], item['path']
    sources[producer] = source
    verification.append(dict(producer=producer, manifest_sha256=sha(source.manifest_bytes), files=len(files)))
    print('VERIFIED', producer, len(files), 'files')
s01, s06 = sources['EDEN-01'], sources['EDEN-06']
lineage = {item['producer']: item['manifest_sha256'] for item in s06.manifest['upstream']}
assert lineage['EDEN-01'] == sha(s01.manifest_bytes), 'EDEN-06 used a different climate contract.'
assert s01.read('data/splits.json') == s06.read('data/splits.json')
splits = json.loads(s01.read('data/splits.json'))
schema = json.loads(s01.read('data/schema_map.json'))
config06 = json.loads(s06.read('models/vision_config.json'))
assert config06['model_sha256'] == sha(s06.read('models/vision.pt')) == s06.manifest['model_sha256']
assert config06['encoder_training_experiment'] == 'AGC4-pretrial-2023'
assert s06.manifest['test_scored'] is False
STEP_MINUTES = int(splits['timestep_minutes'])
STEP = pd.Timedelta(minutes=STEP_MINUTES)
assert STEP_MINUTES == 5 and LOOKBACK_STEPS <= splits['max_lookback_steps']
assert all(h % STEP_MINUTES == 0 for h in HORIZON_MINUTES)
HORIZON_STEPS = [h // STEP_MINUTES for h in HORIZON_MINUTES]
assert max(HORIZON_STEPS) <= splits['max_horizon_steps']
MAX_IMAGE_AGE_MINUTES = (LOOKBACK_STEPS - 1) * STEP_MINUTES
(ROOT / 'data/splits.json').write_bytes(s01.read('data/splits.json'))
(ROOT / 'data/schema_map.json').write_bytes(s01.read('data/schema_map.json'))
save_json(ROOT / 'reports/input_verification.json', verification)

## Cell 3 — Check keys, timestamps, splits, and image features

A key identifies one observation. Climate keys combine experiment, compartment, and UTC timestamp. We recompute split labels from the frozen rules, rather than trusting a column name. The encoder stays frozen and no model weights are loaded.

In [ ]:
def read_table(source, name):
    """Read a verified Parquet table directly from its input bundle."""
    return pd.read_parquet(io.BytesIO(source.read(name)))

climate = read_table(s01, 'data/climate.parquet')
controls = read_table(s01, 'data/controls.parquet')
weather = read_table(s01, 'data/weather.parquet')
vision = read_table(s06, 'vision/vision_features.parquet')
for table in [climate, controls, weather, vision]:
    table['ts_utc'] = pd.to_datetime(table.ts_utc, utc=True).astype('datetime64[ns, UTC]')
    if 'compartment_id' in table:
        table['compartment_id'] = table.compartment_id.astype(str)
    assert table.experiment_id.eq(splits['experiment_id']).all()
for table in [climate, controls]:
    assert not table.duplicated(KEYS).any() and table.ts_utc.eq(table.ts_utc.dt.floor('5min')).all()
assert not weather.duplicated(['experiment_id', 'ts_utc']).any()
assert vision.image_id.is_unique and not vision.duplicated(KEYS).any(), 'Resolve simultaneous camera views before alignment.'

def expected_splits(table):
    """Apply the unchanged primary and chronological rules to any timestamp table."""
    mapping = {str(comp): label for label, comps in splits['primary']['compartments'].items() for comp in comps}
    primary = table.compartment_id.map(mapping)
    assert primary.notna().all()
    rules = splits['chronological']
    chronological = pd.Series('embargo', index=table.index, dtype='object')
    chronological.loc[table.ts_utc < pd.Timestamp(rules['train_end_exclusive'])] = 'train'
    validation = table.ts_utc.ge(pd.Timestamp(rules['validation_start_inclusive'])) & table.ts_utc.lt(pd.Timestamp(rules['validation_end_exclusive']))
    chronological.loc[validation] = 'validation'
    chronological.loc[table.ts_utc >= pd.Timestamp(rules['test_start_inclusive'])] = 'test'
    return primary, chronological

for table in [climate, controls, vision]:
    primary, chronological = expected_splits(table)
    assert table['split'].eq(primary).all() and table.chronological_split.eq(chronological).all()
for _, group in climate.groupby(['experiment_id', 'compartment_id']):
    assert group.sort_values('ts_utc').ts_utc.diff().dropna().eq(STEP).all(), 'Climate grid has a gap.'
EMBEDDINGS = sorted(c for c in vision if c.startswith('embedding_'))
PROXIES = sorted(c for c in vision if c.startswith('proxy_'))
assert len(EMBEDDINGS) == config06['embedding_size'] == 128
assert vision.feature_origin.eq('frozen_encoder_disjoint_2023').all()
assert vision.model_sha256.eq(config06['model_sha256']).all()
assert not vision.transfer_validated.any() and not vision.metric_geometry_verified.any()
usable = vision.usable_model_feature
assert np.isfinite(vision.loc[usable, EMBEDDINGS + PROXIES].to_numpy()).all()
assert vision.loc[~usable, EMBEDDINGS + PROXIES].isna().all().all()
assert vision.valid_depth_fraction.between(0, 1).all()
# Only schema/quality checks are performed here; no holdout labels are used for model selection.
print('Climate rows:', len(climate), '| image rows:', len(vision), '| usable images:', int(usable.sum()))

## Cell 4 — Combine recorded climate and controls; preserve missingness

Weather is already aligned in EDEN-01; we verify that join. Actuator measurements and requested setpoints keep different names. Missing or suspect values are retained in the raw table, with explicit validity flags. No future weather or action readings are made into input features.

In [ ]:
CLIMATE_FIELDS = ['air_temp_c', 'rh_pct', 'co2_ppm', 'par_inside_umol_m2_s', 'humidity_deficit_g_m3']
WEATHER_FIELDS = ['outside_temp_c', 'outside_rh_pct', 'outside_vapor_g_m3', 'radiation_w_m2', 'wind_speed_m_s', 'outside_par_umol_m2_s']
ACTUATOR_FIELDS = ['vent_lee_pct', 'vent_wind_pct', 'heating_pipe_temp_c', 'lighting_pct',
                   'irrigation_flow_duration_min', 'energy_screen_pct', 'blackout_screen_pct']
SETPOINT_FIELDS = ['heating_setpoint_c', 'ventilation_setpoint_c', 'lighting_setpoint_pct',
                   'co2_setpoint_ppm', 'irrigation_interval_setpoint_min']
CONTROL_FIELDS = ACTUATOR_FIELDS + SETPOINT_FIELDS + ['co2_actuation_code']
mapped_fields = {item['canonical'] for item in schema['fields']}
assert set(CLIMATE_FIELDS + WEATHER_FIELDS + CONTROL_FIELDS) <= mapped_fields

# Confirm the stored weather observations match exact source timestamps, including missingness.
weather_check = climate[KEYS + WEATHER_FIELDS].merge(
    weather[['experiment_id', 'ts_utc'] + WEATHER_FIELDS], on=['experiment_id', 'ts_utc'],
    how='left', suffixes=('_climate', '_source'), validate='many_to_one')
for field in WEATHER_FIELDS:
    assert np.allclose(weather_check[field + '_climate'], weather_check[field + '_source'], equal_nan=True)
del weather_check
control_columns = CONTROL_FIELDS + [field + suffix for field in CONTROL_FIELDS for suffix in ['_observed', '_qc_ok']]
base = climate.merge(controls[KEYS + control_columns], on=KEYS, how='left', validate='one_to_one', indicator=True)
assert base['_merge'].eq('both').all() and len(base) == len(climate)
base = base.drop(columns='_merge').rename(columns={'split': 'primary_split'})
base = base.sort_values(KEYS).reset_index(drop=True)
base['source_row_id'] = np.arange(len(base), dtype=np.int64)
for field in CLIMATE_FIELDS + WEATHER_FIELDS + CONTROL_FIELDS:
    base[field + '__valid'] = (base[field + '_observed'].fillna(False).astype(bool)
        & base[field + '_qc_ok'].fillna(False).astype(bool) & np.isfinite(base[field]))
# The publisher defines 1=on, 2=off/out. Keep this categorical meaning instead of scaling 1/2 as a magnitude.
base['co2_is_on'] = base.co2_actuation_code.map({1.0: 1.0, 2.0: 0.0})
base['co2_is_on__valid'] = base.co2_actuation_code__valid & base.co2_is_on.notna()
minutes = base.ts_utc.dt.hour * 60 + base.ts_utc.dt.minute
day = base.ts_utc.dt.dayofyear - 1 + minutes / 1440
for name, angle in [('utc_time', 2 * np.pi * minutes / 1440), ('year_time', 2 * np.pi * day / 365.25)]:
    for trig, function in [('sin', np.sin), ('cos', np.cos)]:
        field = name + '_' + trig
        base[field] = function(angle)
        base[field + '__valid'] = True
TIME_FIELDS = ['utc_time_sin', 'utc_time_cos', 'year_time_sin', 'year_time_cos']
HISTORY_FEATURES = CLIMATE_FIELDS + WEATHER_FIELDS + ACTUATOR_FIELDS + SETPOINT_FIELDS + ['co2_is_on'] + TIME_FIELDS
assert len(HISTORY_FEATURES) == len(set(HISTORY_FEATURES))
# The forecast-weather table has no verified issue-time/lead-time contract; it is intentionally unused.
print('History feature candidates:', len(HISTORY_FEATURES), '| raw invalid values kept with masks')

## Cell 5 — Attach past images separately for each protocol

A backward join finds the latest image that already existed. It runs within the same experiment, compartment, and split. We never skip a newer bad frame to silently reuse an older good one. Old/rejected image features become missing; their IDs and reasons remain for inspection.

In [ ]:
VISION_EMBEDDINGS = ['vision_' + name for name in EMBEDDINGS]
VISION_NUMERIC = VISION_EMBEDDINGS + ['vision_age_minutes', 'vision_valid_depth_fraction']
VISION_FLAGS = ['vision_present', 'vision_usable', 'vision_quality_rejected']
PROTOCOLS = {'primary': 'primary_split', 'chronological': 'chronological_split'}
parts = []
image_audit = vision.rename(columns={
    'split': 'primary_split', 'ts_utc': 'image_ts_utc', 'image_id': 'latest_image_id',
    'usable_model_feature': 'source_image_usable', 'qc_flags': 'image_qc_flags',
    'green_mask_fraction': 'audit_green_mask_fraction',
    **{name: 'audit_' + name for name in PROXIES},
    **{name: 'vision_' + name for name in EMBEDDINGS}})
for protocol, split_column in PROTOCOLS.items():
    for (experiment, compartment, split), left in base.groupby(['experiment_id', 'compartment_id', split_column], sort=True):
        right = image_audit[(image_audit.experiment_id == experiment)
            & (image_audit.compartment_id == compartment) & (image_audit[split_column] == split)].copy()
        if split == 'embargo':
            right = right.iloc[:0]  # Embargo rows remain visible but have no image/model inputs.
        right = right.drop(columns=['experiment_id', 'compartment_id', 'primary_split', 'chronological_split'])
        joined = pd.merge_asof(left.sort_values('ts_utc'), right.sort_values('image_ts_utc'),
                               left_on='ts_utc', right_on='image_ts_utc', direction='backward', allow_exact_matches=True)
        joined['protocol'], joined['split'] = protocol, split
        joined['image_age_minutes_audit'] = (joined.ts_utc - joined.image_ts_utc).dt.total_seconds() / 60
        has_image = joined.latest_image_id.notna()
        assert joined.loc[has_image, 'image_age_minutes_audit'].ge(0).all()
        joined['vision_present'] = has_image & joined.image_age_minutes_audit.le(MAX_IMAGE_AGE_MINUTES)
        joined['vision_stale'] = has_image & ~joined.vision_present
        joined['source_image_usable'] = joined.source_image_usable.fillna(False).astype(bool)
        joined['vision_usable'] = joined.vision_present & joined.source_image_usable
        joined['vision_quality_rejected'] = joined.vision_present & ~joined.source_image_usable
        joined['vision_age_minutes'] = joined.image_age_minutes_audit.where(joined.vision_present)
        joined['vision_valid_depth_fraction'] = joined.valid_depth_fraction.where(joined.vision_present)
        joined.loc[~joined.vision_usable, VISION_EMBEDDINGS] = np.nan
        # Trait proxies are traceability-only columns, never default model inputs.
        joined.loc[~joined.vision_usable, ['audit_' + p for p in PROXIES]] = np.nan
        for field in VISION_NUMERIC:
            joined[field + '__valid'] = np.isfinite(joined[field]) & (
                joined.vision_usable if field in VISION_EMBEDDINGS else joined.vision_present)
        if split == 'embargo':
            assert not joined.vision_present.any()
        parts.append(joined)
aligned = pd.concat(parts, ignore_index=True).sort_values(['protocol'] + KEYS).reset_index(drop=True)
aligned['row_id'] = np.arange(len(aligned), dtype=np.int64)
aligned[VISION_EMBEDDINGS] = aligned[VISION_EMBEDDINGS].astype(np.float32)
assert len(aligned) == len(base) * len(PROTOCOLS)
assert not aligned.duplicated(['protocol'] + KEYS).any()
assert aligned.loc[~aligned.vision_usable, VISION_EMBEDDINGS].isna().all().all()
# Check source timestamps, groups and split labels independently against the original image index.
matched = aligned[aligned.latest_image_id.notna()]
source_lookup = vision.set_index('image_id').loc[matched.latest_image_id]
assert np.array_equal(source_lookup.compartment_id.to_numpy(), matched.compartment_id.to_numpy())
assert np.array_equal(source_lookup.experiment_id.to_numpy(), matched.experiment_id.to_numpy())
for protocol, column in [('primary', 'split'), ('chronological', 'chronological_split')]:
    mask = matched.protocol.eq(protocol).to_numpy()
    assert np.array_equal(source_lookup[column].to_numpy()[mask], matched['split'].to_numpy()[mask])
assert (matched.image_ts_utc <= matched.ts_utc).all()
print('Aligned rows:', len(aligned), '| two separate evaluation protocols')

## Cell 6 — Build history and target window indices

Each row in the index points to 72 consecutive history readings and one target time. Inputs and all intervening target timestamps stay inside one compartment and split. Temperature, humidity, and CO₂ must be valid across the history and at the endpoint. Missing vision never removes a climate example.

In [ ]:
window_parts, rejected_counts = [], []
for (protocol, experiment, compartment, split), group in aligned.groupby(
        ['protocol', 'experiment_id', 'compartment_id', 'split'], sort=True):
    if split == 'embargo':
        continue
    group = group.sort_values('ts_utc')
    row_ids = group.row_id.to_numpy(np.int64)
    assert len(group) < 2 or (np.diff(row_ids) == 1).all(), 'Group rows must be contiguous in aligned.parquet.'
    assert group.ts_utc.diff().dropna().eq(STEP).all(), 'A forecast would cross a time gap.'
    valid_core = group[[target + '__valid' for target in TARGETS]].all(axis=1).to_numpy()
    # Prefix sums make it cheap to detect any invalid reading in the complete history.
    invalid_prefix = np.r_[0, np.cumsum(~valid_core)]
    for horizon in HORIZON_STEPS:
        positions = np.arange(LOOKBACK_STEPS - 1, max(LOOKBACK_STEPS - 1, len(group) - horizon))
        history_good = (invalid_prefix[positions + 1] - invalid_prefix[positions - LOOKBACK_STEPS + 1]) == 0
        target_good = valid_core[positions + horizon]
        keep = history_good & target_good
        selected = positions[keep]
        block = pd.DataFrame(dict(protocol=protocol, experiment_id=experiment, compartment_id=compartment, split=split,
            horizon_steps=horizon, horizon_minutes=horizon * STEP_MINUTES,
            history_start_row=row_ids[selected - LOOKBACK_STEPS + 1], origin_row=row_ids[selected],
            target_row=row_ids[selected + horizon]))
        block['origin_ts_utc'] = group.ts_utc.iloc[selected].to_numpy()
        block['target_ts_utc'] = group.ts_utc.iloc[selected + horizon].to_numpy()
        block['vision_usable_at_origin'] = group.vision_usable.iloc[selected].to_numpy()
        window_parts.append(block)
        rejected_counts.append(dict(protocol=protocol, compartment_id=compartment, split=split,
            horizon_minutes=horizon * STEP_MINUTES, after_boundary_check=len(positions),
            rejected_bad_history=int((~history_good).sum()),
            rejected_bad_endpoint_after_good_history=int((history_good & ~target_good).sum()), kept=int(keep.sum())))
windows = pd.concat(window_parts, ignore_index=True)
windows['window_id'] = np.arange(len(windows), dtype=np.int64)
assert len(windows) and not windows.duplicated(['protocol', 'origin_row', 'horizon_steps']).any()
assert windows.target_row.sub(windows.origin_row).eq(windows.horizon_steps).all()
assert windows.origin_row.sub(windows.history_start_row).eq(LOOKBACK_STEPS - 1).all()
assert (pd.to_datetime(windows.target_ts_utc, utc=True) - pd.to_datetime(windows.origin_ts_utc, utc=True)).eq(
    pd.to_timedelta(windows.horizon_minutes, unit='min')).all()
for pointer in ['history_start_row', 'origin_row', 'target_row']:
    lookup = aligned.iloc[windows[pointer].to_numpy()]
    for key in ['protocol', 'experiment_id', 'compartment_id', 'split']:
        assert np.array_equal(lookup[key].to_numpy(), windows[key].to_numpy())
origin_ids = windows.loc[windows.vision_usable_at_origin, 'origin_row'].unique()
assert (aligned.loc[origin_ids, 'image_ts_utc'] >= aligned.loc[origin_ids, 'ts_utc'] - (LOOKBACK_STEPS - 1) * STEP).all()
pd.DataFrame(rejected_counts).to_csv(ROOT / 'reports/window_rejections.csv', index=False)
display(windows.groupby(['protocol', 'split', 'horizon_minutes']).size().unstack('horizon_minutes'))

## Cell 7 — Fit scalers using training data only

A scaler puts inputs on comparable numeric ranges. We fit one per protocol. Image statistics use each eligible training image once, so a photo is not counted dozens of times just because it was carried forward. Raw missing values stay missing; a model later uses normalized zero plus a separate validity mask.

In [ ]:
def fit_statistics(frame, field):
    """Estimate normalization from valid training observations only."""
    values = frame.loc[frame[field + '__valid'], field].to_numpy(dtype=np.float64)
    assert np.isfinite(values).all()
    if len(values) == 0:
        return dict(n=0, mean=None, scale=None, constant=False, active=False, reason='no valid training observations')
    mean, std = float(values.mean()), float(values.std(ddof=0))
    constant = std <= 1e-12
    return dict(n=len(values), mean=mean, scale=1.0 if constant else std,
                constant=constant, active=not constant, reason='constant in training' if constant else 'active')

scaler = dict(schema_version='1.0', fitted_on='training split only', protocols={},
    transform='(value - mean) / scale where valid; normalized 0 where invalid, always paired with __valid mask',
    raw_missing_values_preserved=True, clipped=False, vision_numeric_scope='origin only; never stack historical image contexts')
for protocol in PROTOCOLS:
    training = aligned[(aligned.protocol == protocol) & (aligned.split == 'train')]
    # Vision embeddings/depth quality are image-level observations; age is a timestamp-level input.
    unique_images = training[training.vision_present].drop_duplicates('latest_image_id')
    stats = {}
    for field in HISTORY_FEATURES + VISION_NUMERIC:
        fit_rows = unique_images if field in VISION_EMBEDDINGS + ['vision_valid_depth_fraction'] else training
        stats[field] = fit_statistics(fit_rows, field)
    history_active = [field for field in HISTORY_FEATURES if stats[field]['active']]
    vision_active = [field for field in VISION_NUMERIC if stats[field]['active']]
    assert set(TARGETS) <= set(history_active)
    scaler['protocols'][protocol] = dict(statistics=stats, history_features=history_active,
        origin_vision_features=vision_active, origin_binary_flags=VISION_FLAGS,
        training_rows=len(training), training_row_ids_sha256=sha(training.row_id.to_numpy('<i8').tobytes()),
        training_image_ids=sorted(unique_images.latest_image_id.tolist()),
        usable_training_image_ids=sorted(unique_images.loc[unique_images.vision_usable, 'latest_image_id'].tolist()))

def normalize(frame, fields, protocol):
    """Return model values AND validity masks; never overwrite the observed data."""
    stats = scaler['protocols'][protocol]['statistics']
    values = np.zeros((len(frame), len(fields)), dtype=np.float32)
    masks = np.zeros_like(values, dtype=bool)
    for j, field in enumerate(fields):
        setting = stats[field]
        assert setting['active'], f'Inactive training feature: {field}'
        valid = frame[field + '__valid'].to_numpy(bool) & np.isfinite(frame[field].to_numpy(float))
        values[valid, j] = ((frame.loc[valid, field].to_numpy(float) - setting['mean']) / setting['scale']).astype(np.float32)
        masks[:, j] = valid
    assert np.isfinite(values).all()
    return values, masks

# Demonstrate the saved normalization contract only on training data, without fitting any model.
for protocol, settings in scaler['protocols'].items():
    training = aligned[(aligned.protocol == protocol) & (aligned.split == 'train')]
    fields = settings['history_features'] + settings['origin_vision_features']
    values, masks = normalize(training.iloc[:256], fields, protocol)
    assert (values[~masks] == 0).all()
    print(protocol, 'history features', len(settings['history_features']), 'vision features',
          len(settings['origin_vision_features']), 'usable training images', len(settings['usable_training_image_ids']))
save_json(ROOT / 'data/scaler.json', scaler)

## Cell 8 — Coverage, contracts, and handoff tables

Coverage tells us how often image information is actually available. Missing vision is a real limitation, not a reason to invent observations. The window contract tells later notebooks exactly which rows may be used as inputs and which are labels.

In [ ]:
coverage = aligned.groupby(['protocol', 'split']).agg(
    rows=('row_id', 'size'), fresh_images=('vision_present', 'sum'), usable_images=('vision_usable', 'sum'),
    rejected_images=('vision_quality_rejected', 'sum'), stale_images=('vision_stale', 'sum')).reset_index()
coverage['usable_fraction'] = coverage.usable_images / coverage.rows
coverage.to_csv(ROOT / 'reports/vision_coverage.csv', index=False)
window_coverage = windows.groupby(['protocol', 'split', 'horizon_minutes']).agg(
    windows=('window_id', 'size'), with_vision=('vision_usable_at_origin', 'sum')).reset_index()
window_coverage.to_csv(ROOT / 'reports/window_coverage.csv', index=False)
missingness = []
for (protocol, split), group in aligned.groupby(['protocol', 'split']):
    for field in HISTORY_FEATURES:
        missingness.append(dict(protocol=protocol, split=split, feature=field,
            rows=len(group), valid=int(group[field + '__valid'].sum())))
pd.DataFrame(missingness).to_csv(ROOT / 'reports/input_missingness.csv', index=False)

feature_contract = dict(schema_version='1.0', experiment_id=splits['experiment_id'], targets=TARGETS,
    target_units={'air_temp_c': 'degree_Celsius', 'rh_pct': 'percent', 'co2_ppm': 'ppm'},
    history_candidates=HISTORY_FEATURES, measured_actuators=ACTUATOR_FIELDS + ['co2_is_on'],
    logged_setpoints=SETPOINT_FIELDS, cyclical_time=TIME_FIELDS,
    optional_origin_vision_candidates=VISION_NUMERIC, origin_vision_flags=VISION_FLAGS,
    default_exclusions=['audit_*', 'vision_stale', 'image_age_minutes_audit', 'co2_actuation_code',
                        'future observed weather', 'future realized controls', '2023 OOF traits'],
    feature_lists='Use protocol-specific active lists in scaler.json; do not select every numeric column.',
    missingness='Every numeric input requires its __valid mask; raw values are not filled or clipped.',
    trait_transfer_validated=False, green_masks_validated=False, uncertainty_calibrated=False,
    image_time_assumption='capture timestamp used as availability timestamp; ingestion/processing latency unknown',
    model_sha256=config06['model_sha256'], encoder_training_experiment='AGC4-pretrial-2023')
window_contract = dict(schema_version='1.0', index_file='data/window_index.parquet', aligned_file='data/aligned.parquet',
    scaler_file='data/scaler.json', lookup='row_id is the zero-based row position in aligned.parquet',
    timestep_minutes=STEP_MINUTES, lookback_steps=LOOKBACK_STEPS,
    history_first_to_last_minutes=MAX_IMAGE_AGE_MINUTES, horizon_minutes=HORIZON_MINUTES,
    history_slice='history_start_row through origin_row, inclusive: history features and their masks only',
    vision_slice='origin_row only: active origin vision features, their masks, and binary flags',
    label_slice='target_row: TARGETS only; never use target-time weather/controls/vision as inputs',
    target_columns=TARGETS, future_covariates='not provided; later operational forecasts must use known plans or explicit persistence assumptions',
    continuity='all history and intervening target ticks remain in one experiment, compartment and split; embargo excluded',
    target_qc='all three targets valid throughout history and at the endpoint; intermediate future labels may be missing and require masks for later trajectory models',
    image_tolerance_minutes=MAX_IMAGE_AGE_MINUTES, image_policy='latest prior image in the same split; no fallback past a newer rejected frame',
    test_policy='test indices are prepared but no test metrics/model selection performed',
    origin_stride_steps=1, overlapping_windows=True,
    uncertainty_note='overlapping windows are dependent; downstream uncertainty should use time blocks rather than independent-row assumptions',
    counts=window_coverage.to_dict(orient='records'))
save_json(ROOT / 'data/feature_contract.json', feature_contract)
save_json(ROOT / 'data/windows.json', window_contract)
aligned.to_parquet(ROOT / 'data/aligned.parquet', index=False)
windows.to_parquet(ROOT / 'data/window_index.parquet', index=False)

fig, axis = plt.subplots(figsize=(9, 4))
development = coverage[coverage['split'].isin(['train', 'validation'])]
axis.bar(development.protocol + ' / ' + development['split'], development.usable_fraction * 100)
axis.set_ylabel('Rows with usable vision (%)')
axis.set_title('Development image coverage within the frozen history')
axis.tick_params(axis='x', labelrotation=15)
fig.tight_layout()
fig.savefig(ROOT / 'reports/vision_coverage.png', dpi=120)
plt.show()
plt.close(fig)
display(coverage)

## Cell 9 — Acceptance checks and named result ZIP

This gate checks alignment and preparation, not forecast accuracy. A sparse-vision status is expected for this small image pilot. Notebook 08 can compare climate-only baselines on all valid windows and experimental vision variants on matched subsets.

In [ ]:
assert aligned.loc[aligned.split.eq('embargo'), 'vision_present'].eq(False).all()
assert windows['split'].isin(['train', 'validation', 'test']).all()
for protocol in PROTOCOLS:
    for split in ['train', 'validation', 'test']:
        for horizon in HORIZON_MINUTES:
            assert ((windows.protocol == protocol) & (windows['split'] == split)
                    & (windows.horizon_minutes == horizon)).any(), (protocol, split, horizon)
    settings = scaler['protocols'][protocol]
    assert not any(name.startswith('audit_') for name in settings['history_features'] + settings['origin_vision_features'])
    training_image_set = set(aligned.loc[(aligned.protocol == protocol) & aligned.split.eq('train')
        & aligned.vision_present, 'latest_image_id'])
    assert set(settings['training_image_ids']) == training_image_set
    assert set(settings['usable_training_image_ids']) <= training_image_set
assert s01.read('data/splits.json') == (ROOT / 'data/splits.json').read_bytes()
# Re-open serialized tables to catch row-order or schema changes before packaging.
saved_keys = pd.read_parquet(ROOT / 'data/aligned.parquet', columns=['row_id', 'protocol'] + KEYS)
assert np.array_equal(saved_keys.row_id.to_numpy(), np.arange(len(aligned)))
assert saved_keys.equals(aligned[['row_id', 'protocol'] + KEYS])
saved_windows = pd.read_parquet(ROOT / 'data/window_index.parquet')
assert saved_windows.equals(windows)

status = 'PASS_ALIGNMENT_SPARSE_VISION_TRANSFER_UNVALIDATED'
report = dict(status=status, aligned_rows=len(aligned), original_climate_rows=len(base), window_rows=len(windows),
    targets=TARGETS, horizons_minutes=HORIZON_MINUTES, image_tolerance_minutes=MAX_IMAGE_AGE_MINUTES,
    test_scored=False, model_trained=False, scaler_fit_split='train only, separately for each protocol',
    proxy_traits_used_as_default_inputs=False, raw_missing_values_preserved=True,
    checks=['upstream hashes/lineage', 'frozen splits', 'no future image matches', 'no cross-split image reuse',
            'contiguous full history/target intervals', 'training-only scalers', 'saved table row order'],
    limitations=['36-image pilot gives sparse eligible vision; many climate windows have no image.',
        'Canopy embeddings are experimental; physical trait proxies and green masks remain unvalidated.',
        'Image capture time is used as availability; operational processing latency is not measured.',
        'Earlier EDEN-04 test summaries were inspected; do not claim an untouched test set.',
        'No future weather/control truth is provided as a model input.',
        'Intermediate future labels may be missing: later trajectory models must use per-step label masks.'],
    next_notebook='EDEN-08 Climate baselines on the fixed aligned windows')
save_json(ROOT / 'reports/alignment_report.json', report)
(ROOT / 'LICENSES.txt').write_text(
    'Upstream 2024 challenge and 2023 pre-trial: Wageningen University & Research / 4TU.ResearchData, CC BY 4.0.\n'
    '2024 DOI: 10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1\n'
    '2023 DOI: 10.4121/e1ee9de9-6ce9-4502-a37c-34b5b1372bed.v1\n'
    'Derived EDEN-06 image features retain source attribution. No raw images or model weights are repackaged.\n', encoding='utf-8')
required = ['data/aligned.parquet', 'data/window_index.parquet', 'data/windows.json', 'data/scaler.json',
    'data/feature_contract.json', 'data/splits.json', 'data/schema_map.json', 'reports/runtime.json',
    'reports/input_verification.json', 'reports/window_rejections.csv', 'reports/vision_coverage.csv',
    'reports/window_coverage.csv', 'reports/input_missingness.csv', 'reports/vision_coverage.png',
    'reports/alignment_report.json', 'LICENSES.txt']
files = []
for relative in required:
    payload = (ROOT / relative).read_bytes()
    files.append(dict(path=relative, size=len(payload), sha256=sha(payload)))
manifest = dict(schema_version='1.0', producer='EDEN-07', code_version='1.0', status=status,
    generated_utc=datetime.now(timezone.utc).isoformat(), upstream=verification, files=files,
    vision_model_sha256=config06['model_sha256'], test_scored=False, model_trained=False,
    metric_geometry_created=False, canopy_transfer_validated=False)
save_json(ROOT / 'manifest.json', manifest)
with zipfile.ZipFile(RESULT, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=3) as bundle:
    for relative in required + ['manifest.json']:
        bundle.write(ROOT / relative, 'eden/' + relative)
with zipfile.ZipFile(RESULT) as bundle:
    assert bundle.testzip() is None
    for item in files:
        payload = bundle.read('eden/' + item['path'])
        assert len(payload) == item['size'] and sha(payload) == item['sha256']
print(status)
print('Return:', RESULT, '| bytes:', RESULT.stat().st_size)

### If a check stops the run

- **Input not found:** attach the EDEN-01 v1.1 and EDEN-06 v1.0 results. Keep one distinct result per producer; ZIP and extracted inputs are supported.
- **Split, hash, or key mismatch:** send the exact error. Do not bypass the check or alter split dates.
- **Low image coverage:** expected with this pilot. Keep missingness flags; do not extend tolerance or interpolate images to make coverage look better.
- **Inactive scaler feature:** that field was constant or absent in training. Later models must use the saved active feature lists; validation/test cannot activate it.

The only file to return is `EDEN-07-Climate-Vision-Alignment-Results-v1.0.zip`. Notebook 08 comes after its result review. This notebook was prepared and statically checked locally; its cells were not executed there.